In [1]:
import copy
import math
import torch
import torch.nn as nn

In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_arch_list())  # 应该包含 sm_120

True
['sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']


## Muti-Header Attention Block

In [3]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [4]:
def attention(query, key, value, mask=None, dropout=None):
    "Compute 'Scaled Dot Product Attention'"
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    p_attn = scores.softmax(dim=-1)
    if dropout is not None:
        p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

In [5]:
class MultiHeadsAttention(nn.Module):

    def __init__(self, d_model, h, dropout=0.1):
        super(MultiHeadsAttention, self).__init__()
        assert d_model % h == 0 # 断言投影矩阵维度可以整除头数
        self.d_model = d_model
        self.h = h
        self.d_k = d_model // h
        self.linears = clones(nn.Linear(d_model, d_model), 4)#QKVO四个线性投影变换
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, query, key, value, mask=None):
        if mask is not None:
            mask = mask.unsqueeze(1)
        nbatches = query.size(0)
        query, key, value = [
            lin(x).view(nbatches, -1, self.h, self.d_k).transpose(1, 2) #拆分多头
            for lin, x in zip(self.linears, (query, key, value))
        ]
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        x = (
            x.transpose(1, 2)
            .contiguous()
            .view(nbatches, -1, self.h * self.d_k)
        )
        del query
        del key
        del value
        return self.linears[-1](x) #WO

## 层归一化
    对每个特征维做归一化，然后做仿射变换，仿射变换让网络自行选择进行归一化的程度，否则任意的输入都会被归一化成正态分布

In [6]:
class LayerNorm(nn.Module):

    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

## 残差连接

In [7]:
class SublayerConnection(nn.Module):

    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        sub_out = sublayer(self.norm(x))
        print(f"[Sublayer] x: {tuple(x.shape)}, sublayer 输出: {tuple(sub_out.shape)}")
        return x + self.dropout(sub_out)

## 位置编码

In [8]:
class PositionalEncoding(nn.Module):
    "Implement the PE function."

    def __init__(self, d_model, dropout, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Compute the positional encodings once in log space.
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)].requires_grad_(False)
        return self.dropout(x)

## 前向网络

In [9]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.w_2(self.dropout(self.w_1(x).relu()))

## Encoder

In [10]:
class EncoderLayer(nn.Module):

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask)) #自注意力塞进残差块
        return self.sublayer[1](x, self.feed_forward)

## Decoder

In [11]:
class DecoderLayer(nn.Module):
    "Decoder is made of self-attn, src-attn, and feed forward (defined below)"

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        "Follow Figure 1 (right) for connections."
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)

## 测试

#### 1. 中英文数据预处理

In [12]:
#词嵌入
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel

# 1. 加载中英对照数据集（HF 上一行代码拉取）
ds = load_dataset("Helsinki-NLP/news_commentary", "en-zh", trust_remote_code=True)
train_data = ds["train"]

# 2. 加载 tokenizer 和词嵌入层
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
hf_model = AutoModel.from_pretrained(model_name)
embedding = hf_model.get_input_embeddings()  # nn.Embedding(词表大小, 768)
d_model = embedding.embedding_dim            # 768，你的 Encoder 要对齐这个值

# 3. 批量编码：文本 → token id 张量
def collate_fn(batch):
    en_texts = [item["translation"]["en"] for item in batch]
    inputs = tokenizer(en_texts, padding=True, truncation=True, return_tensors="pt")
    return inputs["input_ids"], inputs["attention_mask"]

# 4. 用 DataLoader 打包
from torch.utils.data import DataLoader

dl = DataLoader(train_data, batch_size=4, collate_fn=collate_fn)

# 5. 取出一个 batch，做词嵌入，得到 Encoder 可识别的张量
for ids, mask in dl:
    # 词嵌入：id → 向量
    with torch.no_grad():
        x = embedding(ids)                    # (B, L) → (B, L, 768)

    # mask 转换：HF 的 1/0 → attention 的 False/True
    key_padding_mask = (mask == 0)
    print("ids:", ids.shape)                  # (4, L)
    print("词向量:", x.shape)                  # (4, L, 768)
    break

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Helsinki-NLP/news_commentary' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ids: torch.Size([4, 74])
词向量: torch.Size([4, 74, 768])


#### 2. 加载编码器

In [4]:
## 因果掩码测试
import torch
subsequent_mask = torch.triu(torch.ones([3,4,4]), diagonal=1).type(
    torch.uint8
)
subsequent_mask

tensor([[[0, 1, 1, 1],
         [0, 0, 1, 1],
         [0, 0, 0, 1],
         [0, 0, 0, 0]],

        [[0, 1, 1, 1],
         [0, 0, 1, 1],
         [0, 0, 0, 1],
         [0, 0, 0, 0]],

        [[0, 1, 1, 1],
         [0, 0, 1, 1],
         [0, 0, 0, 1],
         [0, 0, 0, 0]]], dtype=torch.uint8)

In [13]:
d_model = x.shape[2]
df = 4 * d_model
padding_mask = (mask == 0).unsqueeze(1).unsqueeze(2)   # (B, 1, 1, L)

self_attn = MultiHeadsAttention(d_model=d_model, h=8)
ffn = FeedForward(d_model=d_model, d_ff=df, dropout=0.1)     # ← df，不是 d_model
encoder = EncoderLayer(size=d_model, self_attn=self_attn, feed_forward=ffn, dropout=0.1)

# 5. 前向
encoder.eval()
for ids, mask in dl:
    with torch.no_grad():
        #词嵌入
        x = embedding(ids)
        #因果掩码
        causal = torch.tril(torch.ones(x.shape[1], x.shape[1])).bool().unsqueeze(0)
        padding = (mask == 0).unsqueeze(1).expand(-1, x.shape[1], -1)   # (B, L, L)
        combined = causal & padding   # (B, L, L)
        print("X:", x.shape) # (B, L, 768)
        print("mask",mask.shape)
        print("combined",combined.shape)
        out = encoder(x, mask=combined)                 # (B, L, 768)
    print("输入:", x.shape, "输出:", out.shape)
    break

X: torch.Size([4, 74, 768])
mask torch.Size([4, 74])
combined torch.Size([4, 74, 74])
[Sublayer] x: (4, 74, 768), sublayer 输出: (4, 74, 768)
[Sublayer] x: (4, 74, 768), sublayer 输出: (4, 74, 768)
输入: torch.Size([4, 74, 768]) 输出: torch.Size([4, 74, 768])
